This notebook is a retrieval demo using a selected trained fold model.
When split="all", the gallery includes all known pairs and is intended for practical demonstration, not for held-out evaluation reporting.

In [ ]:
import os
from pathlib import Path

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from omegaconf import OmegaConf

from src.dataset import MonogramPairDataset
from src.models import DualEncoder

: 

In [ ]:
SELECTED_FOLD = 1
DEMO_SPLIT = "all"   # use "all" for practical demo, "test" for honest fold-only demo
TOP_K_DEFAULT = 10

CHECKPOINT_PATH = "multirun/2026-03-06/15-35-33_dinov3H+_FT_ClipLoss/1/checkpoints/fold_1/best_model.pth"
INDEX_SAVE_PATH = f"precomputed_indices/dinov3H+_fold{SELECTED_FOLD}_{DEMO_SPLIT}.pt"

cfg = OmegaConf.create({
    "model": {
        "name": "dinov3_H+_FT",
        "model_version": "facebook/dinov3-vith16plus-pretrain-lvd1689m",
        "embed_dim": 256,
        "freeze_backbone": True,
        "unfreeze_last_layer": True,
        "proj_head_complexity": 1,
    },
    "data": {
        "data_dir": "/scratch/mahantas/datasets/MonogramSchema_Seal_pairs",
        "splits_dir": "/scratch/mahantas/cross_modal_retrieval/splits",
        "batch_size": 32,   # for notebook / CPU
        "num_workers": 4,
        "shuffle": False,
        "transform": True,
        "fold": SELECTED_FOLD,
    },
    "seed": 7,
})

In [ ]:

index_data  = torch.load(INDEX_SAVE_PATH, map_location="cpu")   

print("Number of pairs:", len(index_data["pair_ids"]))
print("Schema embedding shape:", index_data["schema_embeddings"].shape)
print("Seal embedding shape:", index_data["seal_embeddings"].shape)

In [ ]:
gallery_df = pd.DataFrame({
    "index": list(range(len(index_data["pair_ids"]))),
    "pair_id": index_data["pair_ids"],
    "schema_path": index_data["schema_paths"],
    "seal_path": index_data["seal_paths"],
})

gallery_df.head(20)

In [ ]:
gallery_df[gallery_df["pair_id"].str.contains("281", case=False, na=False)]

In [ ]:
def retrieve(index_data, query_modality="seal", query_idx=0, top_k=10):
    schema_emb = index_data["schema_embeddings"]   # [N, D]
    seal_emb = index_data["seal_embeddings"]       # [N, D]

    if query_modality == "seal":
        q = seal_emb[query_idx]                    # [D]
        sims = torch.matmul(schema_emb, q)         # [N]
        ranked = torch.argsort(sims, descending=True)[:top_k]
        gt_idx = query_idx
        query_path = index_data["seal_paths"][query_idx]
        gallery_paths = index_data["schema_paths"]
        gallery_modality = "schema"

    elif query_modality == "schema":
        q = schema_emb[query_idx]
        sims = torch.matmul(seal_emb, q)
        ranked = torch.argsort(sims, descending=True)[:top_k]
        gt_idx = query_idx
        query_path = index_data["schema_paths"][query_idx]
        gallery_paths = index_data["seal_paths"]
        gallery_modality = "seal"

    else:
        raise ValueError("query_modality must be 'schema' or 'seal'")

    return {
        "query_modality": query_modality,
        "gallery_modality": gallery_modality,
        "query_idx": query_idx,
        "query_path": query_path,
        "ranked_indices": ranked.tolist(),
        "scores": sims[ranked].tolist(),
        "gt_idx": gt_idx,
        "gallery_paths": gallery_paths,
        "pair_ids": index_data["pair_ids"],
    }

In [ ]:
def show_retrieval(result, top_k=10):
    query_img = Image.open(result["query_path"]).convert("RGB")

    fig, axes = plt.subplots(1, top_k + 1, figsize=(3 * (top_k + 1), 4))

    axes[0].imshow(query_img)
    axes[0].set_title(f"Query\n{result['query_modality']}")
    axes[0].axis("off")

    for i, idx in enumerate(result["ranked_indices"][:top_k]):
        img = Image.open(result["gallery_paths"][idx]).convert("RGB")
        axes[i + 1].imshow(img)

        title = f"Rank {i+1}\nscore={result['scores'][i]:.3f}\nID={result['pair_ids'][idx]}"
        if idx == result["gt_idx"]:
            title += "\nGT"

        axes[i + 1].set_title(title, fontsize=10)
        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
QUERY_MODALITY = "seal"   # "seal" or "schema"
QUERY_INDEX = 0
TOP_K = 10

result = retrieve(
    index_data=index_data,
    query_modality=QUERY_MODALITY,
    query_idx=QUERY_INDEX,
    top_k=TOP_K,
)

show_retrieval(result, top_k=TOP_K)

In [ ]:
result_top1 = retrieve(
    index_data=index_data,
    query_modality="seal",
    query_idx=0,
    top_k=1,
)

show_retrieval(result_top1, top_k=1)

In [ ]:
import random

random_indices = random.sample(range(len(index_data["pair_ids"])), 5)

for idx in random_indices:
    print(f"\nQuery index: {idx}, pair_id: {index_data['pair_ids'][idx]}")
    result = retrieve(index_data, query_modality="seal", query_idx=idx, top_k=5)
    show_retrieval(result, top_k=5)